# 0501 · 01 · Ingesta de Parquet con CTAS y COPY INTO

**Del fichero de origen a una tabla Bronze en Databricks.**

En este laboratorio cargas un histórico de usuarios guardado en Parquet de tres formas distintas y comparas qué hace cada una:

| Paso | Técnica | ¿Crea una tabla? | ¿Recuerda lo ya cargado? |
|---|---|---|---|
| 1 | Consultar el fichero (`parquet.` / `read_files`) | No | — |
| 2 | **CTAS** (`CREATE TABLE AS SELECT`) | Sí, de una vez | No: recarga todo |
| 4 | **PySpark** (`spark.read` + `saveAsTable`) | Sí, de una vez | No: recarga todo |
| 5–6 | **COPY INTO** | Carga en una tabla existente | **Sí**: salta los ficheros ya cargados |

## Objetivos de aprendizaje

Al terminar serás capaz de:

- Explorar ficheros de un volumen y consultarlos sin crear una tabla.
- Materializar un histórico en una tabla Delta con CTAS y con PySpark.
- Leer `DESCRIBE TABLE EXTENDED` para saber qué tipo de tabla has creado.
- Cargar datos con `COPY INTO`, resolver un error de esquema con `mergeSchema` y comprobar que la carga es **idempotente**.

## Antes de empezar

1. Ejecuta **`00_preparar_entorno`** (Run all). Copia los cuatro Parquet a `/Volumes/lab0501/upload/upload/users-historical`.
2. Conecta este notebook a **Serverless**.

### Los datos

Cuatro ficheros Parquet con el mismo esquema y **251.501 filas** en total:

| Columna | Tipo | Contenido |
|---|---|---|
| `user_id` | STRING | Identificador del usuario, por ejemplo `U000000001` |
| `user_first_touch_timestamp` | BIGINT | Primer contacto del usuario, en **microsegundos** desde 1970 (UTC) |
| `email` | STRING | Correo ficticio; puede ser nulo |

Todas las tablas del laboratorio se crean en el schema `lab0501.bronze`:

In [ ]:
%sql
USE CATALOG lab0501;
USE SCHEMA bronze;

SELECT current_catalog() AS catalogo, current_schema() AS schema;

## 1 · Explorar los ficheros

Una ruta de volumen tiene la forma `/Volumes/<catálogo>/<schema>/<volumen>/<carpeta>`. El volumen da acceso gobernado a ficheros, pero **todavía no hay ninguna tabla**: solo ficheros.

Primero mira qué hay en la carpeta. Deberías ver cuatro ficheros `part-0000X.snappy.parquet`.

In [ ]:
%sql
LIST '/Volumes/lab0501/upload/upload/users-historical';

### Consultar el Parquet directamente

Parquet guarda el esquema dentro del propio fichero y organiza los datos por columnas, así que Spark puede leerlo sin un `CREATE TABLE` previo.

`LIMIT 10` solo limita lo que se muestra: no modifica los ficheros ni crea nada. Sin `ORDER BY`, no esperes ver siempre las mismas diez filas.

In [ ]:
%sql
SELECT *
FROM parquet.`/Volumes/lab0501/upload/upload/users-historical`
LIMIT 10;

### Consultar con `read_files`

[`read_files`](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files) es la forma recomendada de leer ficheros desde SQL: indicas el formato y opciones, y el lector infiere el esquema.

Compara las columnas con la consulta anterior. Aparece una columna extra, **`_rescued_data`**: recoge los datos que no encajan en el esquema inferido. Que exista la columna no significa que haya errores; fíjate en que aquí vale `NULL` en todas las filas.

In [ ]:
%sql
SELECT *
FROM read_files(
    '/Volumes/lab0501/upload/upload/users-historical',
    format => 'parquet'
)
LIMIT 10;

Antes de cargar nada, cuenta cuántas filas hay en origen. Es la cifra con la que compararás cada carga: **251.501**.

In [ ]:
%sql
SELECT count(*) AS filas
FROM read_files('/Volumes/lab0501/upload/upload/users-historical', format => 'parquet');

## 2 · Cargar el histórico con CTAS

**CTAS** significa `CREATE TABLE AS SELECT`: crear una tabla con el resultado de una consulta. Aquí materializas el origen en una tabla Delta de la capa Bronze, con las columnas tal como las interpreta el lector.

Consultar el fichero no persistía nada. Este paso sí escribe los datos en una tabla propia, independiente de los ficheros de origen.

`CREATE OR REPLACE` hace que la celda se pueda repetir: cada ejecución reconstruye la tabla completa a partir del origen. Los Parquet del volumen no se tocan.

In [ ]:
%sql
CREATE OR REPLACE TABLE historical_users_bronze_ctas_rf
USING DELTA
AS SELECT *
FROM read_files(
    '/Volumes/lab0501/upload/upload/users-historical',
    format => 'parquet'
);

SELECT count(*) AS filas
FROM historical_users_bronze_ctas_rf;

Comprueba que la tabla tiene **251.501 filas**, las mismas que el origen.

## 3 · Inspeccionar qué has creado

Ejecuta la celda y busca estas filas en la sección **Detailed Table Information**:

- **Type = MANAGED**: Unity Catalog gestiona el almacenamiento y los metadatos de la tabla. No has indicado `LOCATION`, por eso es gestionada; leer desde un volumen no convierte el destino en una tabla externa.
- **Provider = delta**: la tabla de destino es Delta, aunque el origen era Parquet. El formato de entrada y el de salida se eligen por separado.
- **Location**: dónde guarda la tabla sus ficheros. No coincide con la ruta del volumen que has leído: **origen y destino son ficheros distintos**.

En una tabla **externa** la ubicación se indica de forma explícita y borrar la tabla no borra sus ficheros. En una tabla **gestionada**, Unity Catalog se encarga del ciclo de vida de los datos al borrarla.

In [ ]:
%sql
DESCRIBE TABLE EXTENDED historical_users_bronze_ctas_rf;

> **Para pensar:** si mañana llega otro fichero a la carpeta, ¿aparece solo en esta tabla?
>
> No. CTAS ejecuta una consulta en un momento concreto; la tabla no queda conectada a la carpeta. Para incorporar el fichero nuevo tendrías que volver a ejecutar el CTAS, que **recarga todo** el histórico. Esto es justo lo que resuelve `COPY INTO` más abajo.

## 4 · La alternativa en PySpark

La misma carga con la API de DataFrames: leer Parquet, escribir una tabla Delta y consultarla por su nombre.

`mode("overwrite")` reemplaza el contenido de la tabla en cada ejecución, igual que el `CREATE OR REPLACE` del CTAS: tampoco recuerda qué ficheros cargó. Se escribe en otra tabla para poder comparar las dos.

In [ ]:
RUTA_ORIGEN = "/Volumes/lab0501/upload/upload/users-historical"
TABLA_PYTHON = "lab0501.bronze.historical_users_bronze_python"

df = spark.read.format("parquet").load(RUTA_ORIGEN)

(df.write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(TABLA_PYTHON))

users_bronze_table = spark.table(TABLA_PYTHON)
users_bronze_table.printSchema()
print("filas:", users_bronze_table.count())
display(users_bronze_table.limit(10))

Comprueba que también hay **251.501 filas**. Fíjate en el esquema: con `spark.read` no aparece `_rescued_data`, que es propia de `read_files`.

## 5 · Ingesta incremental con `COPY INTO`

[`COPY INTO`](https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into) carga los ficheros de una ubicación en una **tabla de Unity Catalog que ya existe**. La operación se puede reintentar y es **idempotente**: los ficheros del origen que ya se cargaron en esa tabla se saltan. Es útil cuando llegan ficheros nuevos a una carpeta y solo quieres cargar esos.

Vas a ver dos ejemplos con los mismos Parquet:

1. **Ejemplo 1:** el error de esquema más habitual y cómo resolverlo.
2. **Ejemplo 2:** preparar la tabla para que el esquema evolucione desde el principio.

### Ejemplo 1 · Error de esquema

1. La celda de abajo crea una tabla vacía, **historical_users_bronze_ci**, con solo dos columnas: **user_id** y **user_first_touch_timestamp**. Pero los Parquet tienen tres: también **email**.

In [ ]:
%sql
CREATE OR REPLACE TABLE historical_users_bronze_ci (
    user_id STRING,
    user_first_touch_timestamp BIGINT
);

2. Intenta cargar los ficheros con `COPY INTO`. **La carga falla a propósito** con el error `COPY_INTO_SCHEMA_MISMATCH_WITH_TARGET_TABLE`: el origen tiene 3 columnas y la tabla solo 2.

   La celda captura el error y lo muestra, para que **Run all** no se detenga aquí. La sentencia que se ejecuta es esta:

   ```sql
   COPY INTO historical_users_bronze_ci
   FROM '/Volumes/lab0501/upload/upload/users-historical'
   FILEFORMAT = PARQUET
   ```

In [ ]:
try:
    spark.sql("""
        COPY INTO lab0501.bronze.historical_users_bronze_ci
        FROM '/Volumes/lab0501/upload/upload/users-historical'
        FILEFORMAT = PARQUET
    """)
    print("La carga no ha fallado: revisa que la tabla se creó solo con dos columnas.")
except Exception as error:
    print("Error esperado:\n")
    print(str(error)[:1500])

   ¿Cómo lo resolverías? Piensa en dos opciones antes de seguir: cambiar la tabla o permitir que la carga la cambie.

3. Añade `COPY_OPTIONS ('mergeSchema' = 'true')`. Con esta opción, `COPY INTO` puede **ampliar el esquema** de la tabla con las columnas nuevas que traigan los ficheros.

   En el resultado, **num_inserted_rows** debe valer **251.501**.

In [ ]:
%sql
COPY INTO historical_users_bronze_ci
FROM '/Volumes/lab0501/upload/upload/users-historical'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');     -- fusiona el esquema de cada fichero con el de la tabla

4. Consulta la tabla. Ahora tiene tres columnas: `email` se ha añadido sola.

In [ ]:
%sql
SELECT *
FROM historical_users_bronze_ci
LIMIT 10;

### Ejemplo 2 · Preparar la evolución de esquema desde el principio

Otra forma de cargar los mismos ficheros: crear una tabla **vacía y sin columnas**, **historical_users_bronze_ci_no_schema**, y dejar que `mergeSchema` la complete en la primera carga. Es útil cuando no conoces el esquema de antemano.

Ejecuta la celda y confirma que **num_inserted_rows** vale **251.501**.

In [ ]:
%sql
-- Borrar la tabla si existe, para poder repetir la demostración
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;

-- Crear una tabla vacía sin indicar el esquema
CREATE TABLE historical_users_bronze_ci_no_schema;

-- COPY INTO crea las columnas a partir de los ficheros
COPY INTO historical_users_bronze_ci_no_schema
FROM '/Volumes/lab0501/upload/upload/users-historical'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

## 6 · Idempotencia: repetir la carga no duplica

`COPY INTO` recuerda qué ficheros ha cargado en cada tabla. Si lo vuelves a ejecutar sin que haya ficheros nuevos en el origen, no carga nada.

1. Ejecuta **otra vez** el mismo `COPY INTO`. Fíjate en que **num_affected_rows**, **num_inserted_rows** y **num_skipped_corrupt_files** valen **0**: los cuatro ficheros ya estaban cargados.

In [ ]:
%sql
COPY INTO historical_users_bronze_ci_no_schema
FROM '/Volumes/lab0501/upload/upload/users-historical'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

2. Comprueba que la tabla sigue teniendo **251.501 filas**, no el doble.

In [ ]:
%sql
SELECT count(*) AS filas
FROM historical_users_bronze_ci_no_schema;

3. Revisa el historial de la tabla. Verás una operación `COPY INTO` por cada ejecución. En **operationMetrics**, la primera tiene `numOutputRows` = 251501. La segunda no escribió filas: según la versión del runtime, aparece con 0 filas o ni siquiera genera una versión nueva.

In [ ]:
%sql
DESCRIBE HISTORY historical_users_bronze_ci_no_schema;

**NOTA:** si llegan ficheros nuevos al origen, `COPY INTO` solo cargará esos. Por eso es una buena opción para un job de **ingesta batch incremental** desde almacenamiento en la nube: no vuelve a leer lo que ya cargó. Si los ficheros llegan de forma continua o en gran número, la herramienta indicada es Auto Loader, que verás en el laboratorio 0502.

Compáralo con el CTAS: si repites la celda del paso 2, la tabla se reconstruye con **todas** las filas de nuevo. El resultado final es el mismo, pero vuelve a leerlo todo.

## 7 · Comprueba lo que has aprendido

1. **¿Qué diferencia hay entre consultar el Parquet y ejecutar un CTAS?** La consulta lee el origen y no guarda nada; el CTAS crea una tabla y escribe en ella el resultado.
2. **¿Por qué la tabla es Delta si el origen es Parquet?** El formato de entrada y el de destino se eligen por separado. Lo has indicado con `USING DELTA` o con `.format("delta")` al escribir.
3. **¿Por qué falla el primer `COPY INTO`?** La tabla no tiene la columna `email` que traen los ficheros, y no se ha permitido la evolución del esquema.
4. **¿Qué cambia al activar `mergeSchema`?** `COPY INTO` puede añadir a la tabla las columnas nuevas del origen.
5. **¿Por qué no aparecen filas nuevas al repetir el `COPY INTO`?** Los ficheros ya se habían cargado en esa tabla y `COPY INTO` los salta.
6. **¿Evita `COPY INTO` cualquier duplicado?** No. Controla **ficheros**, no claves: si llega un fichero nuevo con un usuario que ya existía, se cargará. Eliminar duplicados por clave es otra regla que hay que aplicar aparte.
7. **¿Qué pasa si ejecutas todo el notebook otra vez?** Las celdas `CREATE OR REPLACE` reconstruyen las tablas desde cero, así que vuelves a ver 251.501 filas. La prueba de idempotencia es repetir **solo** la celda del paso 6.1.

## Si algo no funciona

| Síntoma | Qué revisar |
|---|---|
| `LIST` falla o la carpeta está vacía | Ejecuta `00_preparar_entorno` |
| No existe el catálogo `lab0501` | Ejecuta `00_preparar_entorno`; comprueba que tienes permiso para crear catálogos |
| Acceso denegado | Permisos sobre el catálogo, el schema y el volumen, y permiso para crear tablas |
| El ejemplo 1 no da error | La tabla `historical_users_bronze_ci` debe tener solo dos columnas: vuelve a ejecutar su `CREATE OR REPLACE` |
| Repetir `COPY INTO` inserta filas | Se ha recreado la tabla entre medias, o han llegado ficheros nuevos al origen |
| CTAS y PySpark no dan el mismo recuento | Han cambiado los ficheros del origen entre una carga y otra: vuelve a ejecutar `00_preparar_entorno` |

## Volver a empezar

Ejecuta `00_preparar_entorno`: borra las cuatro tablas y deja los Parquet como al principio. Las tablas no se borran al terminar este notebook, para que puedas seguir inspeccionándolas.

## Recursos adicionales

- [read_files](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)
- [CREATE TABLE \[USING\] (incluye CTAS)](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-table-using)
- [COPY INTO (sintaxis)](https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into)
- [Empezar con COPY INTO](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/copy-into/)
- [Tablas gestionadas en Unity Catalog](https://docs.databricks.com/aws/en/tables/managed)